# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**Evidence map**

- Lecture 01 (command line): In Task 1.1 I counted the lines in all three CSVs at once from the terminal with `wc -l data/*.csv`, and used `>` to save the output to `output/file_inventory.txt` instead of just printing it.

- Lecture 02 (Git, functions, files): In Task 1.2 I wrote a `profile(name, table)` function so I could run the same row, column, and duplicate counts on every file in a loop, then saved the results to `output/table_profile.csv`. I committed my work in stages as I finished each section.

- Lecture 03 (environments, NumPy): I ran everything in a `uv` virtual environment so the package versions match `uv.lock`. For the NumPy checkpoint, my array is `(27149, 2)`: each row is a patient with both readings, and the two columns are baseline and week12. I used `mean(axis=0)` because I wanted to average down the patients and get one mean per visit (about 143.0 vs. 137.2 mmHg); `axis=1` would have averaged each patient's two visits together, which isn't what I'm comparing.

- Lecture 04 (Jupyter, labeled pandas data): In Task 4.2 I used `.loc` to keep only the `current` site rows and just the `site_code` and `site_name` columns. Throughout the notebook I used `display()` and `value_counts()` to check what the data looked like before and after each step.

- Lecture 05 (contracts, cleaning, validation): Some visit dates were typed with slashes or missing zeros, so in Task 3.2 I only kept a date if it matched `YYYY-MM-DD` exactly (`str.fullmatch`) and was a real calendar date (`pd.to_datetime(..., errors="coerce")`). In Task 5.1 I put my checks in a Series and used `assert` so the extract can't save if anything fails.

- Lecture 06 (joins, reshaping, concatenation): I stacked the north and south visit files with `pd.concat`, reshaped A1c from wide to long with `melt()` and SBP from long to wide with `pivot()`, and added `validate=` to each merge in Task 4.4 so a repeated key would throw an error instead of quietly duplicating rows.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
# TODO: write a function, such as profile(name, table), that returns the four values in README
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.

def profile(name, table):
    return{
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "exact_duplicate_rows": table.duplicated().sum()
    }

results = []
for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

table_profile = pd.DataFrame(results)
display(table_profile)
table_profile.to_csv(PROFILE_PATH, index=False)

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**Data contract**

What one row should mean in each cleaned table, and what in the raw data breaks that:

- `patients_clean`: one row per registered patient, keyed by `patient_id`. The raw `patients.parquet` has 45 exact duplicate rows, so I drop those first.

- `sites_current`: one row per current site, keyed by `site_code`. In `sites.csv`, `N01` and `S02` each show up twice, once as `current` and once as `retired`, so I keep only the current record. `W03` is listed but has no patients yet, and `E04` shows up in the registry but isn't in the site list at all.

- `visits_clean`: one row per patient per visit after stacking the north and south files, keyed by `patient_id` + `visit`. Each raw visit file has 35 exact duplicate rows. There are also rechecks, where the same patient and visit were measured twice at different times. Some of these only show up after I map spellings like `wk12` to `week12`. For a recheck, I keep the later reading.

- `a1c_wide`: one row per patient with their baseline and week12 A1c results side by side, keyed by `patient_id`. It has no repeated rows or repeated IDs, so nothing needs to be removed.

- `a1c_long`: one row per patient per visit after melting `a1c_wide`, keyed by `patient_id` + `visit`. I drop the rows with no result, so every row has an A1c value.

**Joins for Task 4.4:**

- visits --> patients: many-to-one. A patient can have up to two visits, but each `patient_id` is only in `patients_clean` once.
- patients --> sites: many-to-one. Lots of patients share a site, and each `site_code` is only in `sites_current` once after removing the retired rows.
- visits --> A1c: one-to-one. Each `patient_id` + `visit` pair shows up at most once on each side. Some visits don't have an A1c result, so those stay missing after the left join.


### 2.2 Stack the two visit files

In [4]:
# TODO: rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).

north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})
north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"
visits_raw = pd.concat([north, south], ignore_index=True)

### 2.3 Audit the raw tables

In [5]:
# TODO: count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and
# count, and save it to AUDIT_PATH.

pat = raw["patients"]
gender = pat["gender"]
gender_stripped = gender.str.strip()
site_normalized = pat["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")

display(visits_raw["visit"].str.strip().str.lower().value_counts())
VISIT_MAP = {"baseline": "baseline", "bl": "baseline",
             "week12": "week12", "wk12": "week12", "week 12": "week12"}
visit_mapped = visits_raw["visit"].str.strip().str.lower().map(VISIT_MAP)

sbp_num = pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce")
date_pattern_ok = visits_raw["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
date_parsed = pd.to_datetime(visits_raw["visit_date"], format="%Y-%m-%d", errors="coerce")
date_ok = date_pattern_ok & date_parsed.notna()

pair_check = pd.DataFrame({"patient_id": visits_raw["patient_id"], "visit": visit_mapped})

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs",
    ],
    "count": [
        ((gender_stripped != "") & ~gender.isin(["F", "M", "X"])).sum(),
        (gender_stripped == "").sum(),
        (pat["birth_year"] == "9999").sum(),
        (site_normalized != pat["site_code"]).sum(),
        raw["sites"].duplicated(subset=["site_code"], keep=False).sum(),
        (~visits_raw["visit"].isin(["baseline", "week12"])).sum(),
        (sbp_num == 999).sum(),
        ((sbp_num != 999) & ((sbp_num < 70) | (sbp_num > 250))).sum(),
        (~date_ok).sum(),
        pair_check.duplicated(subset=["patient_id", "visit"], keep=False).sum(),
    ],
})
display(issue_audit)

issue_audit.to_csv(AUDIT_PATH, index=False)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

visit
baseline    29259
week12      26913
bl            455
wk12          381
week 12       363
Name: count, dtype: int64[pyarrow]

,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,380


## Task 3: Clean

### 3.1 Clean the patients

In [6]:
# TODO: list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.

display(raw["patients"]["gender"].value_counts(dropna=False))

GENDER_MAP = {
    "f": "F", "female": "F", "woman": "F",
    "m": "M", "male": "M", "man": "M",
    "x": "X", "nonbinary": "X", "non-binary": "X", "nb": "X",
}

patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates(keep="first")
patients_clean["gender"] = (patients_clean["gender"].str.strip().str.lower().map(GENDER_MAP).astype("category"))
patients_clean["birth_year"] = (patients_clean["birth_year"].replace("9999", pd.NA).astype("Int64"))
patients_clean["site_code"] = (patients_clean["site_code"].str.upper().str.replace("-", "").str.replace(" ", ""))
patients_clean = patients_clean.reset_index(drop=True)



gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]

In [7]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.

display(patients_clean["gender"].cat.categories)

pc = patients_clean
patients_summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max",
               "site_code values", "patients at N01", "patients at S02", "patients at E04"], 
    "value": [len(pc), pc["birth_year"].isna().sum(), pc["birth_year"].min(), pc["birth_year"].max(),
              pc["site_code"].nunique(), (pc["site_code"] == "N01").sum(), (pc["site_code"] == "S02").sum(), (pc["site_code"] == "E04").sum()],
})
display(patients_summary)
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
display(gender_counts)
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

Index(['F', 'M', 'X'], dtype='str')

,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [8]:
# TODO: build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.
visits_clean = None

visits_clean = visits_raw.copy(deep=True)
visits_clean = visits_clean.drop_duplicates(keep="first")

display(visits_clean["visit"].value_counts())
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(VISIT_MAP)

sbp = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = sbp.where(sbp.between(70, 250)).astype("Int64")

visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")

exact_text = visits_clean["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
parsed = pd.to_datetime(visits_clean["visit_date"], format="%Y-%m-%d", errors="coerce")
visits_clean["visit_date"] = parsed.where(exact_text)


visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]

In [9]:
# TODO: step 6, show the repeated patient_id and visit pairs, then keep the later reading.

repeats = visits_clean[visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)]
display(repeats.sort_values(["patient_id", "visit", "visit_time"]))

visits_clean = (visits_clean.sort_values("visit_time").drop_duplicates(subset=["patient_id", "visit"], keep="last"))

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


In [10]:
# TODO: step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).

visits_clean = (visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)
                [["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]])
display(visits_clean["visit"].value_counts(dropna=False))

vc = visits_clean
visits_summary = pd.DataFrame({
    "metric": ["visits", "baseline visits", "week12 visits",
               "visit_date missing", "visit_date min", "visit_date max",
               "sbp_mmhg missing", "sbp_mmhg min", "sbp_mmhg max",
               "weight_kg missing", "weight_kg mean",
               "visits from visits_north.csv", "visits from visits_south.csv"],
    "value": [len(vc), (vc["visit"] == "baseline").sum(), (vc["visit"] == "week12").sum(),
              vc["visit_date"].isna().sum(), vc["visit_date"].min(), vc["visit_date"].max(),
              vc["sbp_mmhg"].isna().sum(), vc["sbp_mmhg"].min(), vc["sbp_mmhg"].max(),
              vc["weight_kg"].isna().sum(), round(vc["weight_kg"].mean(), 2),
              (vc["source_file"] == "visits_north.csv").sum(),
              (vc["source_file"] == "visits_south.csv").sum()],
})
display(visits_summary)
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)

visit
baseline    29599
week12      27582
Name: count, dtype: int64

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [11]:
# TODO: one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decision_table = pd.DataFrame({
    "table": ["patients", "patients", "patients", "patients",
              "visits", "visits", "visits", "visits", "sites"],
    "issue": ["exact duplicate rows",
              "gender spellings that are not a code",
              "site codes needing normalization", 
              "birth_year sentinel 9999",
              "weight column named wt_kg in visits_south.csv",
              "sbp sentinel 999 and readings outside 70 through 250",
              "impossible and loosely written visit dates",
              "repeated patient_id and visit pairs",
              "repeated site_code rows"],
    "decision": [
        "Dropped exact repeats with drop_duplicates(), keeping the first copy.",
        "Stripped and lowercased, then mapped each spelling to F/M/X; unknown and blank became missing; stored as a category.",
        "Uppercased and removed hyphens and spaces so every code reads like N01, S02, or E04.",
        "Replaced 9999 with missing and stored birth_year as Int64.",
        "Renamed wt_kg to weight_kg before stacking the two visit files.",
        "Converted to numbers and kept only 70 through 250 mmHg; 999 and out-of-range values became missing.",
        "Kept a date only when it was exact YYYY-MM-DD text for a real calendar date; anything else became missing.",
        "After mapping visit labels, kept the later reading by visit_time for each patient and visit.",
        "Kept only current site records, one per site_code, for the join.",
    ],
    "reason": [
        "Identical rows are the same patient entered twice and would repeat patient_id in the joins.",
        "Staff typed the same three genders many ways (woman, Male, NB), and unknown does not identify a code.",
        "The same site was typed as n-01, N 01, or S-02, which would split one site into several codes.",
        "9999 means not recorded, and keeping it would distort birth_year min, max, and ages.",
        "The south export used a different column name, so concat would have left two half-empty weight columns.",
        "999 means not measured and values outside 70 to 250 are not plausible clinic readings.",
        "Slash dates, unpadded months, and impossible dates cannot be read reliably as YYYY-MM-DD.",
        "These are rechecks taken minutes apart, and the later reading is the final measurement for that visit.",
        "N01 and S02 each have a retired name, which would duplicate patients in a many-to-one site join.",
    ],
})

decision_table.to_csv(DECISION_LOG_PATH, index=False)

decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,"Dropped exact repeats with drop_duplicates(), ...",Identical rows are the same patient entered tw...
1,patients,gender spellings that are not a code,"Stripped and lowercased, then mapped each spel...",Staff typed the same three genders many ways (...
2,patients,site codes needing normalization,Uppercased and removed hyphens and spaces so e...,"The same site was typed as n-01, N 01, or S-02..."
3,patients,birth_year sentinel 9999,Replaced 9999 with missing and stored birth_ye...,"9999 means not recorded, and keeping it would ..."
4,visits,weight column named wt_kg in visits_south.csv,Renamed wt_kg to weight_kg before stacking the...,"The south export used a different column name,..."
5,visits,sbp sentinel 999 and readings outside 70 throu...,Converted to numbers and kept only 70 through ...,999 means not measured and values outside 70 t...
6,visits,impossible and loosely written visit dates,Kept a date only when it was exact YYYY-MM-DD ...,"Slash dates, unpadded months, and impossible d..."
7,visits,repeated patient_id and visit pairs,"After mapping visit labels, kept the later rea...","These are rechecks taken minutes apart, and th..."
8,sites,repeated site_code rows,"Kept only current site records, one per site_c...","N01 and S02 each have a retired name, which wo..."


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = raw["a1c_wide"].melt(id_vars=["patient_id"],
                                value_vars=["a1c_baseline", "a1c_week12"],
                                var_name="visit",
                                value_name="a1c_pct")
a1c_long["visit"] = a1c_long["visit"].map({"a1c_baseline": "baseline", "a1c_week12": "week12"})
print(a1c_long["a1c_pct"].dtype)
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

a1c_summary = pd.DataFrame({
    "visit": ["baseline", "week12"],
    "results": [(a1c_long["visit"] == v).sum() for v in ["basline", "week12"]],
    "a1c_pct_mean": [round(a1c_long.loc[a1c_long["visit"] == v, "a1c_pct"].mean(), 2)
                     for v in ["baseline", "week12"]]
})
display(a1c_summary)
a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)

float64


,visit,results,a1c_pct_mean
0,baseline,0,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [13]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]

try:
    patients_clean.merge(sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)

display(sites[sites.duplicated(subset=["site_code"], keep=False)])
sites_current = sites.loc[sites["record_status"] == "current", ["site_code", "site_name"]]
display(sites_current)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
3,S02,South Valley Clinic
4,W03,West Clinic


In [14]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

visit_a1c = visits_clean.merge(a1c_long, on=["patient_id", "visit"], how="outer", indicator=True)
merge_counts = visit_a1c["_merge"].value_counts()
display(merge_counts)

merge_audit = pd.DataFrame({
    "check": ["visits with an A1c result", "visits without an A1c result",
              "A1c results without a visit", "visits whose patient is not in the registry",
              "registry patients with no visit",
              "registry patients whose site is not in the site list",
              "site list rows removed by the current-record rule"],
    "count": [merge_counts["both"], merge_counts["left_only"], merge_counts["right_only"],
              (~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum(),
              (~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum(),
              (~patients_clean["site_code"].isin(sites["site_code"])).sum(),
              len(sites) - len(sites_current)],
})
display(merge_audit)
merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64

,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [15]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = (visits_clean.pivot(index="patient_id", columns="visit", values="sbp_mmhg").reset_index())
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]

sbp_wide_sample = sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)]
display(sbp_wide_sample)
sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index=False)


,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
3,P00004,142,<NA>,<NA>
87,P00088,152,147,-5
259,P00260,70,138,68
445,P00446,250,144,-106
486,P00487,<NA>,143,<NA>
849,P00851,<NA>,129,<NA>
2336,P02354,127,125,-2
3789,P03836,134,125,-9
5684,P05763,161,151,-10


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

complete_pairs = sbp_wide[["baseline", "week12"]].dropna()
bp = np.array(complete_pairs, dtype=float)
display(bp.shape)
display(bp.mean(axis=0))

(27149, 2)

array([143.01337066, 137.15061328])

### 4.4 Join the analysis table

In [17]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.

analysis = visits_clean.merge(patients_clean, on="patient_id", how="left", validate="many_to_one")
analysis = analysis.merge(sites_current, on="site_code", how="left", validate="many_to_one")
analysis = analysis.merge(a1c_long, on=["patient_id", "visit"], how="left", validate="one_to_one")

analysis = analysis[["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "a1c_pct",
                     "gender", "birth_year", "site_code", "site_name", "source_file"]]
analysis = analysis.astype({
    "patient_id": "string", "visit": "string", "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64", "weight_kg": "float64", "a1c_pct": "float64",
    "gender": "string", "birth_year": "Int64",
    "site_code": "string", "site_name": "string", "source_file": "string",
})
display(analysis.dtypes)

patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

present_sbp = analysis["sbp_mmhg"].dropna()
present_site = analysis["site_name"].dropna()

checks = pd.Series({
    "analysis rows equal visits_clean rows": len(analysis) == len(visits_clean),
    "no repeated patient_id and visit pair": not analysis.duplicated(subset=["patient_id", "visit"]).any(),
    "sbp_mmhg within 70 through 250": bool(present_sbp.between(70, 250).all()),
    "site_name is a current site": bool(present_site.isin(sites_current["site_name"]).all()),
    "sbp_wide patient_id unique": sbp_wide["patient_id"].is_unique,
    "raw tables unchanged": all(raw[name].equals(raw_snapshot[name]) for name in raw),
})
display(checks)
assert checks.all(), checks[~checks]

analysis rows equal visits_clean rows    True
no repeated patient_id and visit pair    True
sbp_mmhg within 70 through 250           True
site_name is a current site              True
sbp_wide patient_id unique               True
raw tables unchanged                     True
dtype: bool

### 5.2 Save the analysis extract

In [19]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
print(len(analysis_sample), visits_clean["patient_id"].isin(SAMPLE_IDS).sum())
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)

28 28


### 5.3 Read the file back

In [20]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.

round_trip = pd.read_csv(ANALYSIS_SAMPLE_PATH, dtype={
    "patient_id": "string", "visit": "string", "sbp_mmhg": "Int64",
    "weight_kg": "float64", "a1c_pct": "float64", "gender": "string", "birth_year": "Int64",
    "site_code": "string", "site_name": "string", "source_file": "string",
}, parse_dates=["visit_date"])
print(round_trip.equals(analysis_sample))

True
